In [ ]:
import os
import sys
import subprocess
import re

# 1. 设置 Hugging Face 镜像，防止网络报错
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

# 2. 安装核心依赖 (如果已经装好，这一步会自动跳过)
print("检查并安装依赖库...")
!pip install -q torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0 --index-url https://download.pytorch.org/whl/cu124
!pip install -q bitsandbytes==0.46.1 transformers==4.51.2 accelerate peft datasets sentencepiece protobuf "huggingface-hub==0.36.0"

# 3. 下载模型基座 (如果已经下载过，会自动跳过)
from huggingface_hub import snapshot_download
print("检查并下载基座模型 RealVisXL V5.0...")
model_id = "SG161222/RealVisXL_V5.0"
local_dir = "/mnt/workspace/RealVisXL_V5.0"
snapshot_download(
    repo_id=model_id,
    local_dir=local_dir,
    ignore_patterns=["*.bin", "*.pth"]
)
print("环境与模型准备完毕！")

In [ ]:
#===============自动修改源码===================
script_path = "/mnt/workspace/code/diffusers/examples/text_to_image/train_text_to_image_lora_sdxl.py" #根据实际修改路径

print(f"正在打补丁: {script_path} ...")

if os.path.exists(script_path):
    with open(script_path, "r", encoding="utf-8") as f:
        content = f.read()

    # 1. 注释掉版本检查
    content = re.sub(r'check_min_version\("0\.44\.0\.dev0"\)', '# check_min_version("0.44.0.dev0")', content)

    # 2. 在最前面注入 torch.backends.cudnn.enabled = False
    if "torch.backends.cudnn.enabled = False" not in content:
        # 找到第一个 import torch 并在后面追加
        content = content.replace("import torch\n", "import torch\ntorch.backends.cudnn.enabled = False # 解决 A10 cuDNN 报错\n", 1)

    with open(script_path, "w", encoding="utf-8") as f:
        f.write(content)
    print("源码补丁已注入完成！")
else:
    print("找不到源码文件，请确认 /mnt/workspace/code/diffusers 是否存在！")

In [ ]:
# ================= 配置路径 =================
SCRIPT_PATH = "/mnt/workspace/code/diffusers/examples/text_to_image/train_text_to_image_lora_sdxl.py" #根据实际修改路径
BASE_MODEL_PATH = "/mnt/workspace/RealVisXL_V5.0" #下载到本地的基座路径
DATASET_DIR = "/mnt/workspace/my_yabi_dataset/images"  # 你的数据集文件夹
OUTPUT_DIR = "/mnt/workspace/realvis-lora-output"     # LoRA保存路径

# ================= 组装命令 =================
cmd = [
    "accelerate", "launch", SCRIPT_PATH,
    f"--pretrained_model_name_or_path={BASE_MODEL_PATH}",
    f"--train_data_dir={DATASET_DIR}",
    "--caption_column=text",
    "--resolution=1024", #显存不够可降低分辨率
    "--center_crop", "--random_flip",
    "--train_batch_size=1", "--gradient_accumulation_steps=4",
    "--gradient_checkpointing", "--use_8bit_adam",
    "--max_train_steps=2000",   # 可根据时间改成 600
    "--learning_rate=1e-4", "--lr_scheduler=cosine", "--lr_warmup_steps=0",
    "--mixed_precision=fp16",  
    "--checkpointing_steps=500",
    "--validation_prompt=a high-quality portrait photo, detailed skin texture, 8k", #提示词（英文）
    "--validation_epochs=10", 
    f"--output_dir={OUTPUT_DIR}",
    "--seed=42",
    "--resume_from_checkpoint=latest" # 自动断点续训
]

# 解决显存碎片
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

print("启动训练，请耐心等待（千万别关掉这个页面）...\n")
process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)

try:
    for line in iter(process.stdout.readline, ''):
        print(line, end='')
except KeyboardInterrupt:
    print("\n！训练被用户中断")
    process.terminate()
finally:
    process.wait()
    print(f"\n训练进程结束，退出代码: {process.returncode}")

In [ ]:
import torch
from diffusers import StableDiffusionXLPipeline, StableDiffusionXLImg2ImgPipeline
from PIL import Image
from IPython.display import display
#=============测试微调模型===============

model_path = "/mnt/workspace/RealVisXL_V5.0"
lora_dir = "/mnt/workspace/realvis-lora-output"
lora_file = "pytorch_lora_weights.safetensors"

# 1. 加载底模与 LoRA
print("正在加载模型...")
pipe = StableDiffusionXLPipeline.from_pretrained(
    model_path, torch_dtype=torch.float16, use_safetensors=True, variant="fp16" 
).to("cuda")
pipe.load_lora_weights(lora_dir, weight_name=lora_file, adapter_name="my_style")
pipe.set_adapters(["my_style"], adapter_weights=[0.7]) 

# 2. 生成全图
prompt = "Medium shot portrait of a beautiful Asian woman, wearing a white silk dress, looking directly at viewer, perfect symmetrical eyes, soft rim lighting, 85mm lens, photorealistic."
neg_prompt = "(worst quality, low quality:1.4), cross-eyed, strabismus, asymmetric eyes, blurry, (slanted eyes:1.5)"

print("正在生成基础构图...")
base_image = pipe(prompt=prompt, negative_prompt=neg_prompt, num_inference_steps=30, guidance_scale=7.0).images[0]
display(base_image)

# 3. 自动修脸 (两阶段法)
print("正在自动放大并修复脸部...")
img2img_pipe = StableDiffusionXLImg2ImgPipeline(**pipe.components).to("cuda")
img2img_pipe.set_adapters(["my_style"], adapter_weights=[0.8]) 

width, height = base_image.size
crop_box = (int(width*0.3), int(height*0.1), int(width*0.7), int(height*0.5))
face_crop = base_image.crop(crop_box).resize((1024, 1024), Image.LANCZOS)

face_prompt = "Extreme close-up portrait of a beautiful Asian woman, large gorgeous almond eyes, double eyelids, looking directly at viewer, sharp focus on pupils, detailed skin texture."
face_neg = "(slanted eyes, small eyes, monolid:1.5), cross-eyed, blurry, deformed, bad eyes."

fixed_face = img2img_pipe(prompt=face_prompt, negative_prompt=face_neg, image=face_crop, strength=0.35, num_inference_steps=30, guidance_scale=7.0).images[0]

# 4. 拼合并展示
fixed_face_resized_back = fixed_face.resize((crop_box[2] - crop_box[0], crop_box[3] - crop_box[1]), Image.LANCZOS)
final_image = base_image.copy()
final_image.paste(fixed_face_resized_back, (crop_box[0], crop_box[1]))

final_image.save("/mnt/workspace/final_result.png")
print("修复完成！图已保存为 final_result.png")
display(final_image)